# Model A — Fire-risk classifier (ML)

**Data is SIMULATED** (`data/landfill_sensors.csv`), disclosed as synthetic — not real sensor measurement.

Trains a scikit-learn RandomForest to predict `fire_next_24h` for each hourly zone reading.
Time split: **first 70 days train, last 20 test** (no shuffling — we never train on the future).

In [1]:
from pathlib import Path
import pandas as pd

# Robust paths: works whether cwd is the repo root or the notebooks/ folder.
HERE = Path.cwd()
root = next((c for c in (HERE, HERE.parent, HERE.parent.parent)
             if (c / 'data' / 'landfill_sensors.csv').exists()), HERE)
DATA = root / 'data' / 'landfill_sensors.csv'
MODELS = root / 'models'
MODELS.mkdir(exist_ok=True)

df = pd.read_csv(DATA, parse_dates=['timestamp'])
print('Rows:', f'{len(df):,}', '| columns:', list(df.columns))

Rows: 25,920 | columns: ['timestamp', 'zone_id', 'zone_lat', 'zone_lon', 'methane_ppm', 'surface_temp_c', 'ambient_temp_c', 'humidity_pct', 'wind_speed_kmh', 'wind_direction_deg', 'days_since_rain', 'fire_next_24h']


In [2]:
# Time split: first 70 days train, last 20 days test
cutoff = df['timestamp'].min() + pd.Timedelta(days=70)
train = df[df['timestamp'] < cutoff]
test = df[df['timestamp'] >= cutoff]
print(f'Cutoff: {cutoff}')
print(f'Train: {len(train):,} rows  |  Test: {len(test):,} rows')
print('Train fire rate: {:.2%}  |  Test fire rate: {:.2%}'.format(
    train['fire_next_24h'].mean(), test['fire_next_24h'].mean()))

FEATURES = ['methane_ppm', 'surface_temp_c', 'ambient_temp_c', 'humidity_pct',
            'wind_speed_kmh', 'wind_direction_deg', 'days_since_rain']
TARGET = 'fire_next_24h'
X_train, y_train = train[FEATURES], train[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]

Cutoff: 2026-06-10 00:00:00
Train: 20,160 rows  |  Test: 5,760 rows
Train fire rate: 6.88%  |  Test fire rate: 5.71%


In [3]:
from sklearn.ensemble import RandomForestClassifier

clf = RandomForestClassifier(
    n_estimators=300, class_weight='balanced', random_state=42, n_jobs=-1)
clf.fit(X_train, y_train)
print('Trained:', clf)

Trained: RandomForestClassifier(class_weight='balanced', n_estimators=300, n_jobs=-1,
                       random_state=42)


In [4]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)

pred = clf.predict(X_test)
print('Accuracy : {:.3f}'.format(accuracy_score(y_test, pred)))
print('Precision: {:.3f}'.format(precision_score(y_test, pred, zero_division=0)))
print('Recall   : {:.3f}'.format(recall_score(y_test, pred, zero_division=0)))
print('F1       : {:.3f}'.format(f1_score(y_test, pred, zero_division=0)))
print('\nConfusion matrix [rows=actual 0/1, cols=pred 0/1]:')
print(confusion_matrix(y_test, pred))
print('\n' + classification_report(y_test, pred, zero_division=0))

Accuracy : 0.952
Precision: 0.578
Recall   : 0.587
F1       : 0.582

Confusion matrix [rows=actual 0/1, cols=pred 0/1]:
[[5290  141]
 [ 136  193]]

              precision    recall  f1-score   support

           0       0.97      0.97      0.97      5431
           1       0.58      0.59      0.58       329

    accuracy                           0.95      5760
   macro avg       0.78      0.78      0.78      5760
weighted avg       0.95      0.95      0.95      5760



In [5]:
import joblib

imp = sorted(zip(FEATURES, clf.feature_importances_), key=lambda x: -x[1])
print('Feature importances:')
for f, v in imp:
    print(f'  {f:20s} {v:.3f}')

out = MODELS / 'fire_model.joblib'
joblib.dump({'model': clf, 'features': FEATURES}, out)
print('\nSaved model ->', out)

Feature importances:
  surface_temp_c       0.377
  methane_ppm          0.287
  ambient_temp_c       0.095
  humidity_pct         0.081
  days_since_rain      0.061
  wind_direction_deg   0.051
  wind_speed_kmh       0.048



Saved model -> C:\Users\Manny\Documents\IBM_Z_hackathon2026\models\fire_model.joblib
